# Notebook 01 — R_HAT Extraction
**Normative Geometry of Language Models: A Cross-Architectural Study**

This notebook extracts the R_HAT normative direction for each model and computes tension scores for all 100 JBB-Behaviors.

**Outputs per model:**
- `{model}_tensions.csv` — tension scores, polarity, var_exp
- `{model}_rhat.npy` — R_HAT vector
- `bootstrap_summary.csv` — stability over 200 bootstrap resamples
- `layers_sensitivity.csv` — AUC at 10 layer fractions
- `spearman.csv` — 9×9 cross-model Spearman matrix

In [ ]:
# ── CELL 0 — CONFIGURATION ────────────────────────────────────────────────────
import os, gc, warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

warnings.filterwarnings("ignore")

# ─── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
MANIFEST_DIR = Path("./results/manifests")   # output directory
JBB_PATH     = Path("./data/jbb_behaviors.csv")
# ─────────────────────────────────────────────────────────────────────────────

MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "distilgpt2":          ("distilgpt2",                              "base",     False),
    "qwen2-0.5b":          ("Qwen/Qwen2-0.5B",                        "base",     True),
    "gemma-3-270m":        ("google/gemma-3-270m",                    "base",     False),
    "tinyllama-1.1b":      ("TinyLlama/TinyLlama-1.1B-Chat-v1.0",   "chat",     False),
    "phi3-mini":           ("microsoft/Phi-3-mini-4k-instruct",       "instruct", False),
    "llama3-8b-base":      ("meta-llama/Meta-Llama-3.1-8B",          "base",     False),
    "llama3-8b-instruct":  ("meta-llama/Meta-Llama-3.1-8B-Instruct", "instruct", False),
    "mistral-7b-base":     ("mistralai/Mistral-7B-v0.1",             "base",     False),
    "mistral-7b-instruct": ("mistralai/Mistral-7B-Instruct-v0.3",    "instruct", False),
}

# Pipeline config
LAYERS_FRAC      = 0.5   # median layer
MAX_TOKENS_ACTS  = 20    # max tokens per refusal/compliance phrase
BOOTSTRAP_ITERS  = 200
DTYPE            = torch.bfloat16

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB free")
print(f"Models: {len(MODELS)}")

## Refusal/Compliance Pairs
15 contrastive token pairs used to define the normative direction.
Short sequences (≤20 tokens) avoid confounding with sentence-level semantics.

In [ ]:
# ── CELL 1 — REFUSAL / COMPLIANCE PAIRS ──────────────────────────────────────
REFUSAL_PHRASES = [
    "I cannot",
    "I can't do that",
    "I'm unable to",
    "I must decline",
    "I will not",
    "I am not able to",
    "I refuse to",
    "That's not something I can",
    "I'm not going to",
    "I cannot assist with",
    "I won't help with",
    "I am unable to provide",
    "I must refuse",
    "This is something I cannot",
    "I cannot and will not",
]

COMPLIANCE_PHRASES = [
    "Sure",
    "Of course",
    "Here is",
    "Certainly",
    "Absolutely",
    "Happy to help",
    "Here's how",
    "Let me explain",
    "Here are the steps",
    "I'll show you",
    "Here is a guide",
    "Step 1:",
    "To do this",
    "The process involves",
    "You can do this by",
]

assert len(REFUSAL_PHRASES) == len(COMPLIANCE_PHRASES) == 15
print(f"Pairs: {len(REFUSAL_PHRASES)}")
for i, (r, c) in enumerate(zip(REFUSAL_PHRASES[:3], COMPLIANCE_PHRASES[:3])):
    print(f"  {i+1}. REFUSAL: '{r}' | COMPLIANCE: '{c}'")

## R_HAT Extraction Functions

In [ ]:
# ── CELL 2 — R_HAT FUNCTIONS ──────────────────────────────────────────────────

def get_layer_idx(model, frac: float) -> int:
    """Return layer index at a given fraction of total depth."""
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        n = len(model.model.layers)
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        n = len(model.transformer.h)
    else:
        n = 32
    return max(0, min(int(frac * n), n - 1))

def mean_pool_hidden(model, tokenizer, text: str, layer_idx: int) -> np.ndarray:
    """Extract mean-pooled hidden state at layer_idx."""
    device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=MAX_TOKENS_ACTS).to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, return_dict=True)
    h = out.hidden_states[layer_idx + 1][0]  # +1 for embedding layer
    return h.mean(dim=0).float().cpu().numpy()

def extract_rhat(model, tokenizer, layer_idx: int) -> dict:
    """
    Extract R_HAT from 15 refusal/compliance pairs via PCA.
    Returns dict with direction, var_exp, and raw difference matrix.
    """
    diffs = []
    for ref, comp in zip(REFUSAL_PHRASES, COMPLIANCE_PHRASES):
        h_ref  = mean_pool_hidden(model, tokenizer, ref,  layer_idx)
        h_comp = mean_pool_hidden(model, tokenizer, comp, layer_idx)
        diffs.append(h_ref - h_comp)

    A = np.array(diffs)  # (15, hidden_size)
    pca = PCA(n_components=1)
    pca.fit(A)

    direction = pca.components_[0]
    var_exp   = float(pca.explained_variance_ratio_[0])

    return {"direction": direction, "var_exp": var_exp, "diffs": A}

def polarity_correction(direction: np.ndarray, behaviors_df: pd.DataFrame,
                        model, tokenizer, layer_idx: int) -> tuple:
    """
    Fix PCA sign ambiguity: project a harmful behavior and check sign.
    Returns (corrected_direction, polarity) where polarity ∈ {-1, +1}.
    """
    # Use the first behavior (highest tension expected)
    sample_text = behaviors_df.iloc[0]["behavior"]
    h = mean_pool_hidden(model, tokenizer, sample_text, layer_idx)
    raw_tension = float(np.dot(h, direction))

    polarity = -1 if raw_tension < 0 else 1
    return direction * polarity, polarity

def compute_tensions(model, tokenizer, behaviors_df: pd.DataFrame,
                     rhat: np.ndarray, layer_idx: int,
                     var_exp: float, polarity: int) -> pd.DataFrame:
    """Compute tension scores for all 100 behaviors."""
    rows = []
    for _, row in behaviors_df.iterrows():
        h = mean_pool_hidden(model, tokenizer, row["behavior"], layer_idx)
        tau = float(np.dot(h, rhat))
        rows.append({
            "track":    row["track"],
            "category": row["category"],
            "behavior": row["behavior"],
            "tension":  tau,
            "polarity": polarity,
            "var_exp":  var_exp,
        })
    return pd.DataFrame(rows)

print("R_HAT functions defined ✓")

In [ ]:
# ── CELL 3 — BOOTSTRAP STABILITY ─────────────────────────────────────────────

def bootstrap_stability(model, tokenizer, layer_idx: int,
                        n_iter: int = BOOTSTRAP_ITERS,
                        seed: int = 42) -> dict:
    """
    Bootstrap stability of R_HAT: resample 15 pairs with replacement,
    recompute direction, measure cosine with original.
    """
    rng = np.random.default_rng(seed)
    result = extract_rhat(model, tokenizer, layer_idx)
    ref_dir = result["direction"]

    cosines = []
    for _ in range(n_iter):
        idx = rng.integers(0, 15, size=15)
        A_boot = result["diffs"][idx]
        pca = PCA(n_components=1)
        pca.fit(A_boot)
        d = pca.components_[0]
        cos = float(abs(np.dot(d, ref_dir) /
                        (np.linalg.norm(d) * np.linalg.norm(ref_dir) + 1e-8)))
        cosines.append(cos)

    return {
        "cosine_mean": np.mean(cosines),
        "cosine_std":  np.std(cosines),
        "cosines":     cosines,
    }

print("Bootstrap function defined ✓")

In [ ]:
# ── CELL 4 — LAYER SENSITIVITY (AUC across depths) ───────────────────────────

def layer_sensitivity_auc(model, tokenizer, behaviors_df: pd.DataFrame,
                          fracs=None) -> pd.DataFrame:
    """
    Compute AUC (top-10 vs bottom-10 behaviors by tension) at each layer depth.
    """
    if fracs is None:
        fracs = [round(f * 0.1, 1) for f in range(10)]  # 0.0 to 0.9

    rows = []
    for frac in fracs:
        layer_idx = get_layer_idx(model, frac)
        result    = extract_rhat(model, tokenizer, layer_idx)
        direction = result["direction"]

        tensions = []
        for _, row in behaviors_df.iterrows():
            h   = mean_pool_hidden(model, tokenizer, row["behavior"], layer_idx)
            tensions.append(float(np.dot(h, direction)))

        tensions = np.array(tensions)
        # Polarity: positive = more harmful
        if tensions.mean() < 0:
            tensions = -tensions

        sorted_idx = np.argsort(tensions)[::-1]
        top10  = sorted_idx[:10]
        bot10  = sorted_idx[-10:]
        labels = np.array([1]*10 + [0]*10)
        scores = np.concatenate([tensions[top10], tensions[bot10]])
        auc    = float(roc_auc_score(labels, scores))

        rows.append({"frac": frac, "layer": layer_idx,
                     "auc": auc, "var_exp": result["var_exp"]})

    return pd.DataFrame(rows)

print("Layer sensitivity function defined ✓")

In [ ]:
# ── CELL 5 — MAIN LOOP ────────────────────────────────────────────────────────

behaviors_df = pd.read_csv(JBB_PATH)
print(f"Behaviors loaded: {len(behaviors_df)}, categories: {behaviors_df['category'].nunique()}")

all_tensions = {}
bootstrap_rows = []

for model_name, (hf_id, group, trust_rc) in MODELS.items():
    out_csv = MANIFEST_DIR / f"{model_name}_tensions.csv"
    out_npy = MANIFEST_DIR / f"{model_name}_rhat.npy"

    if out_csv.exists() and out_npy.exists():
        print(f"[SKIP] {model_name} — already complete")
        all_tensions[model_name] = pd.read_csv(out_csv)
        continue

    print(f"\n{'='*55}\n  {model_name}  [{group}]\n  {hf_id}")

    # Load model with rope_scaling fix for Phi-3
    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=trust_rc)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    cfg = AutoConfig.from_pretrained(hf_id, trust_remote_code=trust_rc)
    if hasattr(cfg, "rope_scaling") and isinstance(cfg.rope_scaling, dict):
        if "type" not in cfg.rope_scaling:
            cfg.rope_scaling["type"] = "linear"

    model = AutoModelForCausalLM.from_pretrained(
        hf_id, config=cfg, torch_dtype=DTYPE, device_map="auto",
        trust_remote_code=trust_rc, low_cpu_mem_usage=True,
    )
    model.eval()
    layer_idx = get_layer_idx(model, LAYERS_FRAC)
    print(f"  Layer: {layer_idx}")

    # Extract R_HAT
    result   = extract_rhat(model, tokenizer=tok, layer_idx=layer_idx)
    rhat_raw = result["direction"]
    var_exp  = result["var_exp"]

    # Polarity correction
    rhat, polarity = polarity_correction(rhat_raw, behaviors_df, model, tok, layer_idx)
    print(f"  var_exp={var_exp:.3f}  polarity={polarity}")

    # Tension scores
    df_tensions = compute_tensions(model, tok, behaviors_df, rhat, layer_idx, var_exp, polarity)

    # Bootstrap
    print("  Bootstrap stability...")
    boot = bootstrap_stability(model, tok, layer_idx)
    bootstrap_rows.append({
        "model":        model_name,
        "group":        group,
        "var_exp":      var_exp,
        "cosine_mean":  boot["cosine_mean"],
        "cosine_std":   boot["cosine_std"],
        "layer_idx":    layer_idx,
    })
    print(f"  Bootstrap: mean={boot['cosine_mean']:.4f}  std={boot['cosine_std']:.4f}")

    # Save
    df_tensions.to_csv(out_csv, index=False)
    np.save(out_npy, rhat)
    all_tensions[model_name] = df_tensions

    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

# Save bootstrap summary
pd.DataFrame(bootstrap_rows).to_csv(MANIFEST_DIR / "bootstrap_summary.csv", index=False)
print("\n✓ All models processed")

In [ ]:
# ── CELL 6 — LAYER SENSITIVITY & SPEARMAN MATRIX ────────────────────────────
# Run after Cell 5. Requires models to be re-loaded (or use cached outputs).
# This cell is optional — skip if you only need tension scores.

# Spearman cross-model matrix
models_done = [m for m in MODELS if (MANIFEST_DIR / f"{m}_tensions.csv").exists()]
n = len(models_done)
sp_matrix = np.zeros((n, n))

for i, m1 in enumerate(models_done):
    for j, m2 in enumerate(models_done):
        df1 = pd.read_csv(MANIFEST_DIR / f"{m1}_tensions.csv").sort_values("track")
        df2 = pd.read_csv(MANIFEST_DIR / f"{m2}_tensions.csv").sort_values("track")
        t1 = df1["tension"].values * df1["polarity"].iloc[0]
        t2 = df2["tension"].values * df2["polarity"].iloc[0]
        t1 = (t1 - t1.mean()) / t1.std()
        t2 = (t2 - t2.mean()) / t2.std()
        sp_matrix[i, j] = spearmanr(t1, t2)[0]

sp_df = pd.DataFrame(sp_matrix, index=models_done, columns=models_done)
sp_df.to_csv(MANIFEST_DIR / "spearman.csv")
print("Spearman matrix saved.")
print(sp_df.round(3).to_string())